# TNBC miRNA–mRNA GAT study, phase 3s: random-pair edge null and a strong planted signal

**Post-hoc follow-up testing, added after the frozen study and phase 3r.** Not a change to the paper or its frozen
analysis. It follows up on two suggestions from Prof. Ahmet Ay about the phase-3r planted control (where joint_mirdb
matched the degree-preserving rewired null and both were slightly above no-cross):

1. a **uniform random-pair** cross-edge null (same number of edges on uniformly drawn miRNA–mRNA pairs, not
   degree-preserving) next to no-cross and the degree-preserving rewired null;
2. a **much stronger planted signal** (mRNA-example amplitude 8 vs 2; edge-aware oracle ≈ 0.90 vs 0.66–0.77),
   with more seeds (8 per fold) and 4 simulated copies of every outer-test example to cut evaluation noise.

Settings, calibration notes and the pass rule: `kaggle/phase3s_config.py` (frozen before any phase-3s fit with
cross edges). Conditions: `zero` (no signal), `base` (phase 3r's strength), `strong` (decisive).

**Pass rule (same structure as phase 3a / 3r):** per variant and signal condition, on at least one modality the
miRDB-edge model beats every null (no-cross, rewired, random) by ≥ 0.03 macro-F1 with one-sided paired t-test
p < 0.05 over the 5 folds, and every |ΔF1| < 0.03 at `zero`.

One kernel per variant (`VARIANT` below): `gat` (frozen paper GAT) or `selfdrop0.25`.
Settings: Internet **on**, accelerator **None (CPU)**. Smoke test first (`SMOKE_TEST = True`).

In [ ]:
import os, subprocess, time
os.environ['MPLBACKEND'] = 'Agg'  # Kaggle sets matplotlib_inline; matplotlib 3.11 rejects it (same fix as phase 2 / 3a / 3r)

VARIANT = 'gat'      # 'gat' (frozen paper GAT) or 'selfdrop0.25'
SMOKE_TEST = False       # True: fold 0, seed 0, decisive + base conditions, all four edge arms. False: everything.
WORKERS = os.cpu_count()     # parallel fits; each fit is single-threaded (frozen setting)
TIME_BUDGET_H = 11.0         # Kaggle sessions stop at 12 h
RESERVE_H = 1.0              # kept for analysis and the zip; no new GAT fit starts after budget - reserve

BRANCH = 'kaggle-phase3s'
REPO_URL = 'https://github.com/heramn/TNBC-miRNA-mRNA-GNN.git'
SRC = '/tmp/TNBC-miRNA-mRNA-GNN'
PY = '/tmp/p3env/bin/python'
WORK = '/kaggle/working'
T0 = time.time()
DEADLINE = T0 + (TIME_BUDGET_H - RESERVE_H) * 3600
os.environ.update(STUDY_RUNS=f'{WORK}/phase3_runs', PHASE3_OUT=f'{WORK}/phase3s_output', STUDY_CACHE='/tmp/study_cache',
                  OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() and check:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

SM = '--smoke' if SMOKE_TEST else ''
print('variant:', VARIANT, '| workers:', WORKERS, '| smoke test:', SMOKE_TEST)

## 1. Clone the repository at `kaggle-phase3s`

In [ ]:
sh(f'rm -rf {SRC} && git clone -q -b {BRANCH} {REPO_URL} {SRC} && git -C {SRC} log -1 --format="%h %s"')

## 2. Python environment (same pins as phase 1 / 2 / 3r)

In [ ]:
sh('pip install -q uv')
sh('uv venv -q --python 3.12 /tmp/p3env')
sh(f'uv pip install -q --python {PY} numpy==2.5.5 pandas==3.1.1 scipy==1.18.1 scikit-learn==1.7.2 statsmodels==0.15.0 '
   'matplotlib==3.11.2 xgboost==3.4.1 pydeseq2==0.6.0 anndata==0.13.4 torch_geometric==2.8.1 pytest')
sh(f'uv pip install -q --python {PY} torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu')
sh(f'{PY} -c "import sys, torch, torch_geometric, sklearn, numpy; print(sys.version.split()[0], torch.__version__, '
   'torch_geometric.__version__, sklearn.__version__, numpy.__version__)"')

## 3. Data, frozen-file tests, edge-null check

`check-edges` shows, per fold, that the rewired null keeps the miRDB degree sequence and node set exactly while the random null does not.

In [ ]:
os.chdir(SRC)
sh(f'{PY} kaggle/phase3s.py data')
sh(f'{PY} -m pytest -q -p no:cacheprovider study/tests')
sh(f'{PY} kaggle/phase3s.py check-edges')

## 4. Design gate (before any edge-using fit)

Graph-free tabular models must stay ≤ 0.43 mean inner-validation balanced accuracy at both signal conditions; joint_nocross is gated at `base` and reported at `strong`. **The notebook stops if the gate fails.**

In [ ]:
sh(f'{PY} kaggle/phase3s.py run --part gate {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
if SMOKE_TEST:
    print('smoke test: gate verdict needs all 5 folds; skipped')
else:
    sh(f'{PY} kaggle/phase3s.py gate')

## 5. Main fits: four edge arms × three conditions × 5 folds × 8 seeds for this variant

In [ ]:
sh(f'{PY} kaggle/phase3s.py run --part main --variant {VARIANT} {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')

## 6. Analysis

In [ ]:
rc = sh(f'{PY} kaggle/phase3s.py analyze --variant {VARIANT}', check=False)
print('analysis', 'done' if rc == 0 else f'failed (exit {rc})')

## 7. Zip the outputs

In [ ]:
import shutil, zipfile
os.chdir(WORK)
z = shutil.make_archive(f'{WORK}/tnbc_phase3s_{VARIANT}', 'zip', WORK, '.')
with zipfile.ZipFile(z) as zf:
    names = [n for n in zf.namelist() if n.startswith('phase3_runs/p3s') and n.endswith('.csv')]
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB,', len(names), 'per-run CSVs')
print(f'total time {(time.time() - T0) / 3600:.2f} h')